# FIFA World Cup Predictor

## Version 11 - Tournament Simulation (first version)

### Goal

This notebook puts everything together and plays the 2026 World Cup **many times** using Monte
Carlo simulation:

- **Version 9** provides the match probabilities (`predict_world_cup_match`).
- **Version 10** provides the tournament structure (48 teams, 12 groups, 72 fixtures, knockout
  skeleton).
- This notebook provides the **simulation engine**.

### Important honesty notes

- Some official data is still missing (exact fixture venues/dates and the exact bracket
  pairings). We keep those as **explicit data gaps** and design the engine so the official data
  can be inserted later **without rewriting the simulation code**.
- Our dataset has no goals, so we do **not** invent goal differences. Group ties use a clearly
  documented **placeholder** tie-break.
- Knockout matches use the simple **draw-splitting approximation** from Version 9, isolated in
  one helper so it can be replaced by a realistic extra-time/penalty model later.
- The results below are **not** a scientifically accurate forecast - they are a demo of the
  simulation machinery.

### Modular design

```text
predict_world_cup_match()   -> probabilities            (Version 9)
simulate_match()            -> samples one outcome
simulate_group()            -> plays a group, builds standings
get_knockout_probabilities()-> 2-team advance probabilities (approximation)
simulate_knockout()         -> produces one advancing team
simulate_tournament()       -> one complete World Cup
run_simulations()           -> repeats many tournaments
```

In [1]:
import time
from collections import Counter

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Section 1 - Rebuild the Version 8/9 model (self-contained)

We repeat the exact preparation and model from Versions 8-9 so this notebook runs standalone.
Nothing about the model, features or split is changed.

In [2]:
df = pd.read_csv("../data/raw/results.csv")
df = df.dropna(subset=['home_score', 'away_score']).reset_index(drop=True)

def get_result(row):
    if row['home_score'] > row['away_score']:
        return 2
    elif row['home_score'] < row['away_score']:
        return 0
    else:
        return 1

df['result'] = df.apply(get_result, axis=1)
df['neutral_encoded'] = df['neutral'].astype(int)

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print("Prepared matches:", len(df))

Prepared matches: 49413


In [3]:
teams_all = pd.concat([df['home_team'], df['away_team']]).unique()
elo_rating = {team: 1500 for team in teams_all}

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating, expected, actual, k=20):
    return rating + k * (actual - expected)

home_elos = []
away_elos = []

for _, row in df.iterrows():
    home_team = row['home_team']
    away_team = row['away_team']
    home_elo = elo_rating[home_team]
    away_elo = elo_rating[away_team]
    home_elos.append(home_elo)
    away_elos.append(away_elo)
    expected_home = expected_score(home_elo, away_elo)
    expected_away = expected_score(away_elo, home_elo)
    if row['result'] == 2:
        actual_home, actual_away = 1, 0
    elif row['result'] == 0:
        actual_home, actual_away = 0, 1
    else:
        actual_home, actual_away = 0.5, 0.5
    elo_rating[home_team] = update_elo(home_elo, expected_home, actual_home)
    elo_rating[away_team] = update_elo(away_elo, expected_away, actual_away)

df['home_elo'] = home_elos
df['away_elo'] = away_elos
df['elo_difference'] = df['home_elo'] - df['away_elo']

print("Elo ratings built.")

Elo ratings built.


In [4]:
FORM_WINDOW = 5
team_form_history = {team: [] for team in teams_all}
home_forms = []
away_forms = []

for _, row in df.iterrows():
    home_history = team_form_history[row['home_team']]
    away_history = team_form_history[row['away_team']]
    home_forms.append(np.mean(home_history) if len(home_history) > 0 else 0.0)
    away_forms.append(np.mean(away_history) if len(away_history) > 0 else 0.0)
    if row['result'] == 2:
        hp, ap = 1.0, 0.0
    elif row['result'] == 0:
        hp, ap = 0.0, 1.0
    else:
        hp, ap = 0.5, 0.5
    home_history.append(hp)
    away_history.append(ap)
    if len(home_history) > FORM_WINDOW:
        home_history.pop(0)
    if len(away_history) > FORM_WINDOW:
        away_history.pop(0)

df['home_form'] = home_forms
df['away_form'] = away_forms
df['abs_elo_difference'] = (df['home_elo'] - df['away_elo']).abs()
df['form_difference'] = df['home_form'] - df['away_form']

print("Form features built.")

Form features built.


In [5]:
split_index = int(len(df) * 0.8)
train = df.iloc[:split_index]
test = df.iloc[split_index:]

feature_cols = [
    'home_elo', 'away_elo', 'elo_difference',
    'home_form', 'away_form', 'neutral_encoded',
    'abs_elo_difference', 'form_difference',
]

X_train, y_train = train[feature_cols], train['result']
X_test, y_test = test[feature_cols], test['result']

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (39530, 8) Test: (9883, 8)


In [6]:
final_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight={0: 1.0, 1: 2.0, 2: 1.0},
)
final_model.fit(X_train, y_train)

sanity = final_model.predict_proba(X_test).argmax(axis=1)
print("Model trained. Test accuracy (matches V8's 0.5412):", round(accuracy_score(y_test, sanity), 4))

Model trained. Test accuracy (matches V8's 0.5412): 0.5412


# Section 2 - Version 9's prediction system

We bring over Version 9's reusable predictor: team name aliases, latest-Elo / latest-form lookup,
the venue rule, and `predict_world_cup_match()`. We add a small **batch** helper
`predict_many(pairs)` that predicts several matches in one go - it returns the same probabilities,
just faster to feed a whole round or group into the model.

In [7]:
TEAM_ALIASES = {
    'USA': 'United States',
    'US': 'United States',
    'U.S': 'United States',
    'U.S.A': 'United States',
    'Czechia': 'Czech Republic',
    'Türkiye': 'Turkey',
    'Turkiye': 'Turkey',
}

def canonical_team(team):
    return TEAM_ALIASES.get(team, team)

def get_latest_elo(team):
    team = canonical_team(team)
    if team not in elo_rating:
        return 1500.0
    return elo_rating[team]

def get_latest_form(team):
    team = canonical_team(team)
    history = team_form_history.get(team, [])
    if len(history) == 0:
        return 0.0
    return float(np.mean(history))

def predict_world_cup_match(team_a, team_b, neutral=True):
    """Return P(A win), P(draw), P(B win) plus the predicted result."""
    probs = predict_many([(team_a, team_b, neutral)])[0]
    return {
        'team_a': team_a,
        'team_b': team_b,
        'team_a_win_probability': probs[2],
        'draw_probability': probs[1],
        'team_b_win_probability': probs[0],
        'predicted_result': {0: 'Team B wins', 1: 'Draw', 2: 'Team A wins'}[int(probs.argmax())],
    }

def build_feature_row(team_a, team_b, neutral):
    home_elo = get_latest_elo(team_a)
    away_elo = get_latest_elo(team_b)
    home_form = get_latest_form(team_a)
    away_form = get_latest_form(team_b)
    return {
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_difference': home_elo - away_elo,
        'home_form': home_form,
        'away_form': away_form,
        'neutral_encoded': 1 if neutral else 0,
        'abs_elo_difference': abs(home_elo - away_elo),
        'form_difference': home_form - away_form,
    }

def predict_many(pairs):
    """
    pairs : list of (team_a, team_b, neutral).
    Returns an array with one row per match; columns are in class order
    [0, 1, 2] = [Team B win, Draw, Team A win].
    """
    frame = pd.DataFrame([build_feature_row(a, b, n) for a, b, n in pairs])
    return final_model.predict_proba(frame)

pred = predict_world_cup_match('Brazil', 'Germany', neutral=True)
print("Sanity check - Brazil vs Germany:")
print(f"  P(Brazil win): {pred['team_a_win_probability']:.3f}   P(draw): {pred['draw_probability']:.3f}   P(Germany win): {pred['team_b_win_probability']:.3f}")

Sanity check - Brazil vs Germany:
  P(Brazil win): 0.560   P(draw): 0.230   P(Germany win): 0.210


# Section 3 - Version 10's 2026 World Cup structure

We bring over the confirmed tournament structure: the 48 teams, the 12 groups (A-L) from the
official final draw, the 72 group fixtures, and the knockout skeleton. Venues/dates that are not
officially confirmed stay as `TBD (official schedule)` - exactly as in Version 10.

In [8]:
GROUPS = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['United States', 'Paraguay', 'Australia', 'Türkiye'],
    'E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

all_teams = sorted({t for tl in GROUPS.values() for t in tl})
print("Teams:", len(all_teams), "| Groups:", len(GROUPS))

Teams: 48 | Groups: 12


In [9]:
PAIRINGS = [(0, 1), (2, 3), (0, 2), (3, 1), (1, 2), (0, 3)]

HOST_HOME_COUNTRY = {'Mexico': 'Mexico', 'United States': 'United States', 'Canada': 'Canada'}
HOST_OPENERS = {
    ('A', 0): 'Mexico City Stadium',
    ('D', 0): 'Los Angeles Stadium',
    ('B', 0): 'Toronto Stadium',
}

def build_group_fixtures():
    fixtures = []
    for group, teams in GROUPS.items():
        for i, (ia, ib) in enumerate(PAIRINGS):
            team_a, team_b = teams[ia], teams[ib]
            venue = HOST_OPENERS.get((group, i), 'TBD (official schedule)')
            if venue.startswith('TBD'):
                neutral, host = True, None
            else:
                host = team_a if HOST_HOME_COUNTRY.get(canonical_team(team_a)) else None
                neutral = host is None
            fixtures.append({
                'match_id': f"G-{group}-{i + 1}",
                'group': group,
                'team_a': team_a,
                'team_b': team_b,
                'venue': venue,
                'neutral': neutral,
                'host_nation_if_applicable': host,
            })
    return fixtures

fixtures = build_group_fixtures()
print("Group-stage fixtures:", len(fixtures))
print("Groups with a confirmed host home match:", len([f for f in fixtures if not f['neutral']]))

Group-stage fixtures: 72
Groups with a confirmed host home match: 3


In [10]:
# Precompute each group's 6 match probabilities ONCE and reuse them in every simulation.
group_meta = {}
for group in GROUPS:
    pairs = [(f['team_a'], f['team_b'], f['neutral']) for f in fixtures if f['group'] == group]
    probas = predict_many(pairs)
    group_meta[group] = (pairs, probas)

print("Group probabilities precomputed.")
g = 'C'
for (a, b, n), p in zip(group_meta[g][0], group_meta[g][1]):
    print(f"{a:8s} vs {b:8s}  P(A)={p[2]:.2f} P(draw)={p[1]:.2f} P(B)={p[0]:.2f}")

Group probabilities precomputed.
Brazil   vs Morocco   P(A)=0.63 P(draw)=0.18 P(B)=0.19
Haiti    vs Scotland  P(A)=0.17 P(draw)=0.29 P(B)=0.55
Brazil   vs Haiti     P(A)=0.84 P(draw)=0.12 P(B)=0.04
Scotland vs Morocco   P(A)=0.10 P(draw)=0.65 P(B)=0.25
Morocco  vs Haiti     P(A)=0.59 P(draw)=0.38 P(B)=0.03
Brazil   vs Scotland  P(A)=0.38 P(draw)=0.57 P(B)=0.06


# Part 1 - One complete tournament simulation

### Group stage sampling

For each match we use `np.random.default_rng().choice()` to sample **one** of the three outcomes
in proportion to the model's probabilities. We never simply take the most likely outcome - that
is what makes this a Monte Carlo simulation.

### Group standings & tie-breaks

Win = 3 pts, Draw = 1 pt, Loss = 0 pts.

Our dataset/model has **no goals**, so we do not invent goal difference. Instead we use a clearly
documented **placeholder tie-break**: when two teams in a group have equal points, the rank is
decided at random (via the simulation's random generator). This will be replaced by the official
FIFA tie-break rules once match-level scoring is available.

In [11]:
def simulate_match(probs, rng):
    """Sample one outcome from (P(team B), P(draw), P(team A)).
    Returns 2 = team A wins, 1 = draw, 0 = team B wins."""
    return int(rng.choice([0, 1, 2], p=probs))

def simulate_group(group_key, pairs, probas, rng):
    """Play all 6 matches of a group; return per-team standings."""
    standings = {t: {'points': 0, 'wins': 0, 'draws': 0, 'losses': 0} for t in GROUPS[group_key]}
    for (team_a, team_b, neutral), probs in zip(pairs, probas):
        outcome = simulate_match(probs, rng)
        if outcome == 2:      # team A wins
            standings[team_a]['points'] += 3
            standings[team_a]['wins'] += 1
            standings[team_b]['losses'] += 1
        elif outcome == 0:    # team B wins
            standings[team_b]['points'] += 3
            standings[team_b]['wins'] += 1
            standings[team_a]['losses'] += 1
        else:                 # draw
            standings[team_a]['points'] += 1
            standings[team_a]['draws'] += 1
            standings[team_b]['points'] += 1
            standings[team_b]['draws'] += 1
    return standings

def rank_group(standings, rng):
    """Rank a group: points descending.
    PLACEHOLDER tie-break (no goals available): equal points are broken at random.
    Returns the team list from 1st to 4th."""
    return sorted(standings, key=lambda t: (-standings[t]['points'], rng.random()))

def rank_third_places(third_items, rng):
    """Rank the 12 third-placed teams across groups; return the 8 best.
    PLACEHOLDER tie-break: points descending, then random."""
    ordered = sorted(third_items, key=lambda x: (-x[2]['points'], rng.random()))
    return [team for (group, team, st) in ordered[:8]]

### Knockout matches

Knockout matches must always produce a winner. We convert the model's three probabilities into a
two-team advance probability with the same simple approximation as Version 9:

```text
P(A advances) = P(A win) + P(draw) * P(A win) / (P(A win) + P(B win))
P(B advances) = P(B win) + P(draw) * P(B win) / (P(A win) + P(B win))
```

This is clearly an **approximation** for extra time and penalties. It is isolated in
`get_knockout_probabilities()` (and a batched sibling) so a better extra-time / penalties model
can replace it later without touching the rest of the simulator.

### Bracket pairing gap

Version 10 marks the exact bracket pairings as **pending the official bracket sheet**. Until that
data is available, the Round of 32 is assembled with a clearly-labeled **placeholder pairing**
(first half vs second half of the ordered qualifiers) and every knockout match is treated as
**neutral**. Swapping in the official bracket only changes how qualifiers are paired - the
rounds themselves are unchanged.

In [12]:
def get_knockout_probabilities(team_a, team_b, neutral=True):
    """
    Convert the model's P(A win), P(draw), P(B win) into P(A advances), P(B advances).
    APPROXIMATION: the draw is split proportionally to the win probabilities.
    REPLACE THIS when we model extra time / penalties properly.
    """
    probs = predict_many([(team_a, team_b, neutral)])[0]
    p_a_win, p_draw, p_b_win = probs[2], probs[1], probs[0]
    split = p_a_win + p_b_win
    if split == 0:
        return 0.5, 0.5
    p_a = p_a_win + p_draw * (p_a_win / split)
    p_b = p_b_win + p_draw * (p_b_win / split)
    return p_a, p_b

def get_knockout_probabilities_many(pairs):
    """Batched version of get_knockout_probabilities for whole rounds."""
    probs = predict_many([(a, b, True) for a, b in pairs])
    p_a_win, p_draw, p_b_win = probs[:, 2], probs[:, 1], probs[:, 0]
    split = p_a_win + p_b_win
    safe = np.where(split > 0, split, 1.0)
    p_a = p_a_win + p_draw * np.where(split > 0, p_a_win / safe, 0.0)
    p_b = 1.0 - p_a
    return p_a, p_b

def simulate_knockout_round(participants, rng):
    """Pair up participants and return the winners of the round."""
    pairs = [(participants[i], participants[i + 1]) for i in range(0, len(participants), 2)]
    p_a_adv, _ = get_knockout_probabilities_many(pairs)
    winners = []
    for (team_a, team_b), p_a in zip(pairs, p_a_adv):
        winners.append(team_a if rng.random() < p_a else team_b)
    return winners

def assemble_round_of_32(ranked_groups, third_teams):
    """
    Build the 32 Round-of-32 participants from 12 winners + 12 runners-up + 8 best thirds.
    PLACEHOLDER pairing: qualifiers are ordered (winners, runners-up, thirds) and paired
    first half vs second half. Replace this with the official bracket sheet when available.
    """
    winners = [ranked_groups[g][0] for g in GROUPS]
    runners = [ranked_groups[g][1] for g in GROUPS]
    participants = winners + runners + third_teams
    # Placeholder pairing: first half vs second half.
    half = len(participants) // 2
    ordered = []
    for i in range(half):
        ordered.append(participants[i])
        ordered.append(participants[i + half])
    return ordered

In [13]:
ROUND_NAMES = ['Round of 32', 'Round of 16', 'Quarter-finals', 'Semi-finals']

def simulate_tournament(random_seed=None, rng=None):
    """Run one complete World Cup. Returns standings, each round's teams and the champion."""
    if rng is None:
        rng = np.random.default_rng(random_seed)

    # --- Group stage ---
    standings = {}
    for group, (pairs, probas) in group_meta.items():
        standings[group] = simulate_group(group, pairs, probas, rng)

    ranked_groups = {g: rank_group(standings[g], rng) for g in GROUPS}

    # --- Qualification ---
    third_items = [(g, ranked_groups[g][2], standings[g][ranked_groups[g][2]]) for g in GROUPS]
    third_teams = rank_third_places(third_items, rng)

    r32 = assemble_round_of_32(ranked_groups, third_teams)

    # --- Knockout stage ---
    participants_by_round = {ROUND_NAMES[0]: r32}
    current = r32
    for name in ROUND_NAMES[1:]:
        current = simulate_knockout_round(current, rng)
        participants_by_round[name] = current

    # Semi-final winners become the finalists (4 -> 2), then the final (2 -> 1).
    finalists = simulate_knockout_round(current, rng)
    champion = simulate_knockout_round(finalists, rng)[0]

    return {
        'standings': standings,
        'ranked_groups': ranked_groups,
        'third_teams': third_teams,
        'r32': r32,
        'participants_by_round': participants_by_round,
        'finalists': finalists,
        'champion': champion,
    }

### Run ONE tournament and inspect it

Before running thousands of simulations we run a single one and print every stage. A fixed seed
makes this single run reproducible.

In [14]:
t0 = time.time()
result = simulate_tournament(random_seed=2026)
runtime_one = time.time() - t0

print("Champion:", result['champion'])
print(f"One tournament simulated in {runtime_one:.3f} seconds.\n")

Champion: Spain
One tournament simulated in 0.146 seconds.



In [15]:
print("=== GROUP STANDINGS ===\n")
for group in GROUPS:
    rows = []
    ranked = result['ranked_groups'][group]
    for pos, team in enumerate(ranked, start=1):
        st = result['standings'][group][team]
        rows.append({'Pos': pos, 'Team': team, 'P': st['points'],
                     'W': st['wins'], 'D': st['draws'], 'L': st['losses']})
    print(f"Group {group}")
    print(pd.DataFrame(rows).to_string(index=False))
    print()

=== GROUP STANDINGS ===

Group A
 Pos         Team  P  W  D  L
   1  South Korea  7  2  1  0
   2       Mexico  5  1  2  0
   3 South Africa  2  0  2  1
   4      Czechia  1  0  1  2

Group B
 Pos                   Team  P  W  D  L
   1                 Canada  9  3  0  0
   2            Switzerland  4  1  1  1
   3 Bosnia and Herzegovina  4  1  1  1
   4                  Qatar  0  0  0  3

Group C
 Pos     Team  P  W  D  L
   1   Brazil  7  2  1  0
   2  Morocco  4  1  1  1
   3 Scotland  3  0  3  0
   4    Haiti  1  0  1  2

Group D
 Pos          Team  P  W  D  L
   1       Türkiye  7  2  1  0
   2      Paraguay  5  1  2  0
   3 United States  2  0  2  1
   4     Australia  1  0  1  2

Group E
 Pos        Team  P  W  D  L
   1     Ecuador  5  1  2  0
   2     Germany  5  1  2  0
   3 Ivory Coast  3  0  3  0
   4     Curaçao  1  0  1  2

Group F
 Pos        Team  P  W  D  L
   1       Japan  7  2  1  0
   2 Netherlands  7  2  1  0
   3      Sweden  3  1  0  2
   4     Tunisia  0  0  0 

In [16]:
print("=== QUALIFIED / ROUND PARTICIPANTS ===\n")
print("Round of 32 participants ({:d}):".format(len(result['r32'])))
print(result['r32'])
print()
for name in ROUND_NAMES[1:]:
    teams = result['participants_by_round'][name]
    print(f"{name} ({len(teams)} teams):")
    print(teams)
    print()
print("Finalists (2):")
print(result['finalists'])
print()
print("Champion (1):")
print([result['champion']])

=== QUALIFIED / ROUND PARTICIPANTS ===

Round of 32 participants (32):
['South Korea', 'Germany', 'Canada', 'Netherlands', 'Brazil', 'Iran', 'Türkiye', 'Spain', 'Ecuador', 'Norway', 'Japan', 'Austria', 'Belgium', 'DR Congo', 'Uruguay', 'Panama', 'France', 'Algeria', 'Argentina', 'England', 'Portugal', 'Bosnia and Herzegovina', 'Croatia', 'Colombia', 'Mexico', 'Sweden', 'Switzerland', 'Ivory Coast', 'Morocco', 'Scotland', 'Paraguay', 'United States']

Round of 16 (16 teams):
['Germany', 'Netherlands', 'Brazil', 'Spain', 'Norway', 'Japan', 'Belgium', 'Panama', 'France', 'Argentina', 'Portugal', 'Croatia', 'Sweden', 'Switzerland', 'Morocco', 'United States']

Quarter-finals (8 teams):
['Germany', 'Spain', 'Japan', 'Belgium', 'Argentina', 'Portugal', 'Switzerland', 'Morocco']

Semi-finals (4 teams):
['Spain', 'Japan', 'Argentina', 'Morocco']

Finalists (2):
['Spain', 'Argentina']

Champion (1):
['Spain']


### Validation checks for one tournament

We verify the structural invariants of a single simulated World Cup.

In [17]:
def validate_simulation(res):
    checks = []
    # 48 teams start, 12 groups, 4 per group
    checks.append(('48 teams start', len(all_teams) == 48))
    checks.append(('12 groups', len(res['standings']) == 12))
    checks.append(('4 teams per group', all(len(s) == 4 for s in res['standings'].values())))

    # knockout sizes
    checks.append(('32 enter Round of 32', len(res['r32']) == 32))
    checks.append(('16 enter Round of 16', len(res['participants_by_round']['Round of 16']) == 16))
    checks.append(('8 enter Quarter-finals', len(res['participants_by_round']['Quarter-finals']) == 8))
    checks.append(('4 enter Semi-finals', len(res['participants_by_round']['Semi-finals']) == 4))
    checks.append(('2 enter Final', len(res['finalists']) == 2))
    checks.append(('1 champion', res['champion'] in all_teams))

    # no team advances twice in any single knockout round
    dup_ok = True
    for teams in res['participants_by_round'].values():
        if len(teams) != len(set(teams)):
            dup_ok = False
    checks.append(('no duplicate advances in any round', dup_ok))

    return checks

report = validate_simulation(result)
all_pass = all(passed for _, passed in report)
pd.DataFrame(report, columns=['Check', 'Passed'])

,Check,Passed
0,48 teams start,True
1,12 groups,True
2,4 teams per group,True
3,32 enter Round of 32,True
4,16 enter Round of 16,True
5,8 enter Quarter-finals,True
6,4 enter Semi-finals,True
7,2 enter Final,True
8,1 champion,True
9,no duplicate advances in any round,True


# Part 2 - Monte Carlo: run 1,000 tournaments

We repeat the whole tournament many times. Each tournament uses its own reproducible seed drawn
from a fixed master seed, so the full run is reproducible. We record each tournament's champion
and aggregate a championship probability table.

```text
Champion Probability = Champion Count / Number of Simulations
```

In [18]:
def run_simulations(n=1000, random_seed=42):
    """Simulate n complete World Cups and count champions."""
    master_rng = np.random.default_rng(random_seed)
    champion_counts = Counter()
    for i in range(n):
        seed = int(master_rng.integers(0, 2 ** 31))
        res = simulate_tournament(random_seed=seed)
        champion_counts[res['champion']] += 1
    return champion_counts

N_SIMS = 1000
t0 = time.time()
champion_counts = run_simulations(N_SIMS, random_seed=42)
runtime_1000 = time.time() - t0

print(f"{N_SIMS} tournaments simulated in {runtime_1000:.1f} seconds.")

1000 tournaments simulated in 135.3 seconds.


In [19]:
rows = [{'Team': t, 'Champion Count': c, 'Champion Probability': c / N_SIMS}
        for t, c in champion_counts.items()]
champion_df = (pd.DataFrame(rows)
               .sort_values('Champion Probability', ascending=False)
               .reset_index(drop=True))

print(f"Teams that won at least one simulation: {len(champion_df)}")
print(f"Total simulations: {champion_df['Champion Count'].sum()}")
print(f"Sum of all championship probabilities: {champion_df['Champion Probability'].sum():.4f}")
print()
champion_df

Teams that won at least one simulation: 32
Total simulations: 1000
Sum of all championship probabilities: 1.0000



,Team,Champion Count,Champion Probability
0,Spain,140,0.140
1,Argentina,126,0.126
2,France,112,0.112
3,Brazil,98,0.098
4,England,74,0.074
5,Colombia,65,0.065
6,Portugal,64,0.064
7,Germany,59,0.059
8,Netherlands,55,0.055
9,Japan,33,0.033


In [20]:
print("=== Top 10 championship probabilities ===\n")
champion_df.head(10).to_string(index=False)

=== Top 10 championship probabilities ===



'       Team  Champion Count  Champion Probability\n      Spain             140                 0.140\n  Argentina             126                 0.126\n     France             112                 0.112\n     Brazil              98                 0.098\n    England              74                 0.074\n   Colombia              65                 0.065\n   Portugal              64                 0.064\n    Germany              59                 0.059\nNetherlands              55                 0.055\n      Japan              33                 0.033'

# Summary

### What works

- One complete tournament simulation runs end-to-end: 72 group matches, standings, qualification
  (12 winners + 12 runners-up + 8 best thirds = 32), then knockouts to a champion.
- All structural validation checks pass for the single run.
- `run_simulations(1000)` produces a reproducible championship-probability table whose
  probabilities sum to ~1.0.
- Group matches sample outcomes **randomly** according to the model probabilities (Monte Carlo,
  not "pick the favourite").

### Explicit data gaps and approximations (do not treat as a forecast)

1. **No goals / goal difference.** We never invent scores, so group tie-breaks use a random
   placeholder instead of FIFA's official rules. A goal model is needed.
2. **Knockout model is an approximation.** Draws are split proportionally to win probability;
   extra time and penalties are not modelled realistically. Isolated in
   `get_knockout_probabilities()` for later replacement.
3. **Bracket pairings are provisional.** The Round of 32 uses a placeholder pairing and all
   knockout matches are neutral, pending the official bracket sheet and knockout venues.
4. **Fixture venues/dates are incomplete.** Only the three host openers have confirmed venues;
   the rest remain `TBD (official schedule)`. Host advantage is therefore only applied to those
   three fixtures.
5. **Third-place match is not simulated**, and a team's *stage reached* is not yet recorded.

### Runtime

- One-simulation and 1,000-simulation runtimes are printed above (see the single-run cell and
  the 1,000-run cell).